# Unit 5 — Installing MCP plugins & skills ⭐

MCP = Model Context Protocol. Think of it as npm for agent tools.

Three plugin types map to three transports:
- **`MCPStdioTool`** — local subprocess
- **`MCPStreamableHTTPTool`** — remote URL
- **`MCPWebsocketTool`** — persistent socket


In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from agent_framework import MCPStdioTool, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient

## Part A — local MCP plugin (stdio)

`uvx mcp-server-calculator` runs a calculator MCP server in an ephemeral env. First cell-run might take ~10s while `uvx` downloads the package.

In [ ]:
async with MCPStdioTool(
    name="calculator",
    command="uvx",
    args=["mcp-server-calculator"],
    load_prompts=False,  # the calculator server only exposes tools, not prompts
) as calc_tool:
    agent = OpenAIChatCompletionClient().as_agent(
        name="MathAgent",
        instructions="Use the calculator tool for any arithmetic.",
        tools=[calc_tool],
    )
    result = await agent.run("What is 23 * 47 + 113?")
    print(result.text)

## Part B — remote MCP plugin (HTTP)

Microsoft runs a public MCP server for their docs at `learn.microsoft.com/api/mcp`. Any agent can query it.

In [ ]:
async with MCPStreamableHTTPTool(
    name="ms-learn",
    url="https://learn.microsoft.com/api/mcp",
    load_prompts=False,  # server only exposes tools
) as learn_tool:
    agent = OpenAIChatCompletionClient().as_agent(
        name="DocsAgent",
        instructions="Answer Microsoft-tooling questions using the Microsoft Learn tool.",
        tools=[learn_tool],
    )
    result = await agent.run("How do I create an Azure resource group with the az CLI?")
    print(result.text)

## Part C — combining plugins

Two MCP plugins + a custom Python function, all on one agent. The agent routes each question to the right tool automatically.

In [ ]:
from pathlib import Path
from typing import Annotated
from pydantic import Field

NOTES = Path("./notes")

def save_note(title: Annotated[str, Field(description="kebab-case filename.")],
              content: Annotated[str, Field(description="Markdown content.")]) -> str:
    """Save a note as a Markdown file."""
    NOTES.mkdir(exist_ok=True)
    p = NOTES / f"{title}.md"
    p.write_text(content, encoding="utf-8")
    return f"saved to {p}"

async with (
    MCPStdioTool(name="calculator", command="uvx", args=["mcp-server-calculator"], load_prompts=False) as calc,
    MCPStreamableHTTPTool(name="ms-learn", url="https://learn.microsoft.com/api/mcp", load_prompts=False) as learn,
):
    agent = OpenAIChatCompletionClient().as_agent(
        name="StudyBuddy",
        instructions="Use calculator for math, ms-learn for Azure/MS questions, save_note to persist summaries.",
        tools=[calc, learn, save_note],
    )
    session = agent.create_session()
    r1 = await agent.run("What is 500 * 23?", session=session); print("1:", r1.text)
    r2 = await agent.run("How do I list Azure Function apps with az?", session=session); print("2:", r2.text)
    r3 = await agent.run("Save that as a note called 'list-az-funcs'.", session=session); print("3:", r3.text)